In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Evaluation of Formulas from First Order Logic

In this notebook, we show how formulas from *first order logic* can be evaluated in OCaml.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from [group theory](https://en.wikipedia.org/wiki/Group_theory).
A group is defined as a triple $ \langle G, \mathrm{e}, \circ \rangle $
where:
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.

Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$
  * $\forall x: \exists y: y \circ x = \mathrm{e}$
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$

A group is *commutative* if, additionally, the following formula is satisfied:
  * $\forall x: \forall y: x \circ y = y \circ x$

We import the types `term` and `formula` and the function `parse` from the notebook `FOL-Parser.ipynb`:
```
type term =
  | Var of string                  (* variable *)
  | Fun of string * term list      (* function application, constants have no arguments *)

type formula =
  | Verum                          (* ⊤ *)
  | Falsum                         (* ⊥ *)
  | Atom   of string * term list   (* atomic formula p(t1, ..., tn) *)
  | Not    of formula              (* ¬ *)
  | And    of formula * formula    (* ∧ *)
  | Or     of formula * formula    (* ∨ *)
  | Imp    of formula * formula    (* → *)
  | Iff    of formula * formula    (* ↔ *)
  | Forall of string * formula     (* ∀x: f *)
  | Exists of string * formula     (* ∃x: f *)
```

In [ ]:
import_notebook "FOL-Parser.ipynb"

We also need the modules `RSet` and `RMap`, which are loaded by `setup.ml`.  `RMap` implements finite maps, i.e. functions with a finite domain.  It is the counterpart of the class `RecursiveMap` of the TypeScript library `recursive-set`.  A map of type `('k, 'v) RMap.t` maps keys of type `'k` to values of type `'v`.  `RMap.of_list` takes a list of pairs of the form `(key, value)` and creates a map.  For example,
```
      RMap.of_list [([0; 0], 0);
                    ([0; 1], 1);
                    ([1; 0], 1);
                    ([1; 1], 0)]
```
creates the map
$$ \bigl\{ \langle 0, 0\rangle \mapsto 0,\;
           \langle 0, 1\rangle \mapsto 1,\;
           \langle 1, 0\rangle \mapsto 1,\;
           \langle 1, 1\rangle \mapsto 0
   \bigr\}.
$$
The keys of this map are lists of numbers.  `RMap.find k m` returns the value that the map `m` associates with the key `k`.

In [ ]:
RMap.find [0; 1] (RMap.of_list [([0; 0], 0); ([0; 1], 1); ([1; 0], 1); ([1; 1], 0)])

Our parser distinguishes between variables and function/predicate symbols as follows:
- *Variables* start with an **upper case** letter (e.g., `X`, `Y`).
- *Functions* and *predicates* start with a **lower case** letter (e.g., `e`, `p`).
- We can use the arithmetic operators `+`, `-`, `*`, `/`, `%`, `**` as infix operators.
- We can use the relational operators `<`, `>`, `=`, `≤`, `≥` as infix operators.
- The logical symbols `⊤`, `⊥`, `∧`, `∨`, `¬`, `→`, `↔`, `∀`, `∃` are supported with the usual precedences.

  The quantifiers `∀`, `∃` bind stronger than `∧` and `∨`.

  **Note** that `∧` binds stronger than `∨`.  However, in order to avoid confusion
  we will not mix `∧` and `∨` in this lesson.

We represent the symbols from group theory as follows:
- The neutral element "$\mathrm{e}$" is the 0-ary function `e`.
- The operator "$\circ$" is represented by the arithmetic operator `*`.
- Equality "$=$" is natively supported by the parser as `=`.

Then the formulas of group theory can be written as shown below.

In [ ]:
let g1 = "∀X: e * X = X"
let g2 = "∀X: ∃Y: Y * X = e"
let g3 = "∀X: ∀Y: ∀Z: (X * Y) * Z = X * (Y * Z)"
let g4 = "∀X: ∀Y: X * Y = Y * X"

In [ ]:
let f1 = parse g1

In [ ]:
let f2 = parse g2

In [ ]:
let f3 = parse g3

In [ ]:
let f4 = parse g4

## A Structure for Group Theory

We use maps to map lists of arguments to the values of functions, and sets of lists of arguments to represent predicates.
To simplify matters, we assume that the universe is a set of numbers.

In [ ]:
type universe                 = int RSet.t
type function_interpretation  = (int list, int) RMap.t
type predicate_interpretation = int list RSet.t

Assume a signature
$$ \Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle $$
is given. A *$\Sigma$-structure* $\mathcal{S}$ is a
pair $\langle \mathcal{U}, \mathcal{J} \rangle$, such that the following holds:
 * $\mathcal{U}$ is a non-empty set. This set is also called the
   *universe* of the $\Sigma$-structure. This universe contains the *values*.
 * $\mathcal{J}$ is the *interpretation* of the function and predicate symbols.

   Formally, we define $\mathcal{J}$ as a mapping with the following properties:
   + Every function symbol $c \in \mathcal{F}$ with $\texttt{arity}(c) = 0$ is mapped to an element
     of $\mathcal{U}$, i.e. we have $\mathcal{J}(c) \in \mathcal{U}$.

     Instead of writing $\mathcal{J}(c)$ we will write $c^{\mathcal{J}}$.
   + Every function symbol $f \in \mathcal{F}$ with $\texttt{arity}(f) = m$ and $m > 0$ is mapped to
     an $m$-ary function
     $$ f^\mathcal{J}\colon \mathcal{U}^m \rightarrow \mathcal{U}$$
     that maps $m$-tuples of the universe $\mathcal{U}$ into the universe $\mathcal{U}$.
   + Every predicate symbol $p \in \mathcal{P}$ with $\texttt{arity}(p) = 0$ is mapped to
     the set $\mathbb{B}$ of truth values, i.e. $p^{\mathcal{J}} \in \{ \mathtt{true}, \mathtt{false}\}$.
   + Every predicate symbol $p \in \mathcal{P}$ with $\texttt{arity}(p) = n$ such that $n > 0$ is mapped to
     a subset
     $$ p^\mathcal{J} \subseteq \mathcal{U}^n. $$

     The idea is that an atomic formula of the form $p(t_1, \cdots, t_n)$
     is interpreted as $\texttt{true}$ exactly if and only if the interpretation of the tuple
     $\langle t_1, \cdots, t_n \rangle$ is an element of the set $p^\mathcal{J}$.
   + If the symbol `=` is a member of the set of predicate symbols $\mathcal{P}$, then the
     interpretation of  the equality symbol `=` has to be *canonical*, i.e. we must have
     $$ =^\mathcal{J} \;=\; \bigl\{ \langle u, u \rangle \mid u \in \mathcal{U} \bigr\}. $$

A $\Sigma$-structure $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ is represented as an OCaml *record* with the
following fields:
* `universe` is the universe $\mathcal{U}$,
* `functions` is a map that maps every function symbol $f$ to a map $f^{\mathcal{J}}$. In turn,
  $f^{\mathcal{J}}$ maps lists of numbers to numbers.
* `predicates` is a map that maps every $n$-ary predicate symbol $p$ to a set $p^{\mathcal{J}}$. In turn,
  $p^{\mathcal{J}}$ is a set of lists of length $n$.

In [ ]:
type structure = {
  universe   : universe;
  functions  : (string, function_interpretation) RMap.t;
  predicates : (string, predicate_interpretation) RMap.t;
}

Assume a signature
$$\Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle$$
is given.  Furthermore,  $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ is a  $\Sigma$-structure.
Then a function of the form
$$ \mathcal{I}: \mathcal{V} \rightarrow \mathcal{U} $$
is an *$\mathcal{S}$ variable assignment*.

This function is represented as a map from variable names to numbers.

In [ ]:
type variable_assignment = (string, int) RMap.t

We define the universe `u` as the set $\{0, 1\}$:

In [ ]:
let u : universe = RSet.of_list [0; 1]

We define interpretations for the constant `e`  and the binary function symbol `*`.  As `e` has no arguments, its interpretation maps the empty list to the value of `e`.

In [ ]:
let neutral_element : function_interpretation = RMap.of_list [([], 0)]

In [ ]:
let product : function_interpretation =
  RMap.of_list [([0; 0], 0);
                ([0; 1], 1);
                ([1; 0], 1);
                ([1; 1], 0)]

The interpretation of `=` is the set $\bigl\{ \langle x, x \rangle \mid x \in \mathcal{U} \bigr\}$.

In [ ]:
let identity : predicate_interpretation = RSet.map (fun x -> [x; x]) u

We proceed to define the interpretation $\mathcal{J}$.  This interpretation is split in the interpretation of the function symbols and the interpretation of the predicate symbols.

In [ ]:
let functions = RMap.of_list [("e", neutral_element); ("*", product)]

In [ ]:
let predicates = RMap.of_list [("=", identity)]

In [ ]:
let s : structure = { universe = u; functions = functions; predicates = predicates }

We define a variable assignment $\mathcal{I}$.  The concrete definition given here is arbitrary, because the axioms for group theory contain no free variables.

In [ ]:
let i : variable_assignment = RMap.of_list [("X", 0); ("Y", 1); ("Z", 0)]

## Evaluation Functions

If $\Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle$ is a signature,
$\mathcal{S} = \langle\mathcal{U},\mathcal{J}\rangle$ is a  $\Sigma$-structure and $\mathcal{I}$ is an  $\mathcal{S}$
variable assignment, then for every term $t$ the *value $\mathcal{S}(\mathcal{I}, t)$*
is defined by induction on $t$:
 + If $x \in \mathcal{V}$ we define:
   $$\mathcal{S}(\mathcal{I}, x) := \mathcal{I}(x). $$
 + If $c \in \mathcal{F}$ and $\texttt{arity}(c) = 0$, then
   $$\mathcal{S}(\mathcal{I}, c) := c^\mathcal{J}. $$
 + If $t$ is a term of the form $f(t_1,\cdots,t_n)$, then we define
   $$\mathcal{S}\bigl(\mathcal{I}, f(t_1,\cdots,t_n)\bigr) :=
                           f^\mathcal{J}\bigl( \mathcal{S}(\mathcal{I}, t_1), \cdots, \mathcal{S}(\mathcal{I}, t_n) \bigr). $$

The function `eval_term` takes three arguments:
 - `t` is a term,
 - `s` is a structure, and
 - `i` is a variable assignment.

It computes the value $\mathcal{S}(\mathcal{I}, t)$, i.e. it evaluates the term `t` in the structure `s` with the variable assignment `i`.  A constant is a function symbol without arguments, so the second and the third case are handled together: The values of the arguments are computed, and the list of these values is looked up in the interpretation of the function symbol.

In [ ]:
let rec eval_term (t : term) (s : structure) (i : variable_assignment) : int =
  match t with
  | Var x -> RMap.find x i
  | Fun (f, args) ->
      let arg_values = List.map (fun arg -> eval_term arg s i) args in
      RMap.find arg_values (RMap.find f s.functions)

The function `modify` takes three arguments:
 - `i` is a variable assignment,
 - `x` is a variable, and
 - `c` is a value.

It computes the modified variable assignment $\mathcal{I}[x/c]$.  Here,
$\mathcal{I}[x/c]$ denotes the variable assignment
that maps the variable  $x$ to the value $c$ and that otherwise agrees with $\mathcal{I}$
$$\mathcal{I}[x/c](y) := \left\{
    \begin{array}{ll}
    c               & \mbox{if}\; y = x;  \\
    \mathcal{I}(y)  & \mbox{otherwise}.          \\
    \end{array}
    \right.
$$
As maps are immutable, `RMap.add` returns a new map and `i` is not changed.

In [ ]:
let modify (i : variable_assignment) (x : string) (c : int) : variable_assignment =
  RMap.add x c i

The function `eval_formula` takes three arguments:
 - `f` is a formula,
 - `s` is a structure, and
 - `i` is a variable assignment.

It computes the value $\mathcal{S}(\mathcal{I}, F)$, i.e. it evaluates the formula `f` in the structure `s` with the variable assignment `i`.

Let $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ be a $\Sigma$-structure and $\mathcal{I}$ a $\mathcal{S}$-variable
assignment.  Then the value $\mathcal{S}(\mathcal{I},F)$ is defined by induction:
  * $\mathcal{S}(\mathcal{I},\top) := \mathtt{true}$ and $\mathcal{S}(\mathcal{I},\bot) := \mathtt{false}$.
  * If $p$ is a propositional variable, then
    $$\mathcal{S}(\mathcal{I}, p) := p^\mathcal{J}.$$
  * $\mathcal{S}\bigl(\mathcal{I}, p(t_1,\cdots,t_n)\bigr) :=
          \Bigl(\bigl\langle \mathcal{S}(\mathcal{I}, t_1), \cdots, \mathcal{S}(\mathcal{I}, t_n) \bigr\rangle \in p^\mathcal{J}\Bigr)$.
  * $\mathcal{S}(\mathcal{I}, \neg F) := \require{enclose} \enclose{circle}{\neg}\bigl(\mathcal{S}(\mathcal{I}, F)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \wedge G) := \require{enclose} \enclose{circle}\wedge\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \vee G) := \require{enclose} \enclose{circle}\vee\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \rightarrow G) := \require{enclose} \enclose{circle}\rightarrow\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \leftrightarrow G) := \require{enclose} \enclose{circle}\leftrightarrow\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}\bigl(\mathcal{I}, \forall x\colon F\bigr) := \left\{
      \begin{array}{ll}
         \mathtt{true}  & \mbox{if}\; \mathcal{S}(\mathcal{I}[x/c], F) = \mathtt{true}\quad \mbox{holds for all}\; c\in \mathcal{U}; \\
         \mathtt{false} & \mbox{otherwise}.
      \end{array}
      \right.$
   * $\mathcal{S}\bigl(\mathcal{I}, \exists x \colon F\bigr) := \left\{
      \begin{array}{ll}
         \mathtt{true}  & \mbox{if}\; \mathcal{S}(\mathcal{I}[x/c], F) = \mathtt{true}\quad \mbox{holds for some}\; c\in \mathcal{U}; \\
         \mathtt{false} & \mbox{otherwise}.
      \end{array}
      \right.$

**Implementation:** For a quantified formula, the local function `eval_at c` evaluates the formula `g` with the modified variable assignment $\mathcal{I}[x/c]$.  `RSet.for_all` and `RSet.exists` apply `eval_at` to the elements of the universe.  A propositional variable is an atomic formula without arguments, so it is handled by the case `Atom`.

In [ ]:
let rec eval_formula (f : formula) (s : structure) (i : variable_assignment) : bool =
  match f with
  | Verum      -> true
  | Falsum     -> false
  | Not g      -> not (eval_formula g s i)
  | And (g, h) -> eval_formula g s i && eval_formula h s i
  | Or  (g, h) -> eval_formula g s i || eval_formula h s i
  | Imp (g, h) -> eval_formula g s i <= eval_formula h s i
  | Iff (g, h) -> eval_formula g s i =  eval_formula h s i
  | Forall (x, g) ->
      let eval_at c = eval_formula g s (modify i x c) in
      RSet.for_all eval_at s.universe
  | Exists (x, g) ->
      let eval_at c = eval_formula g s (modify i x c) in
      RSet.exists eval_at s.universe
  | Atom (p, args) ->
      let arg_values = List.map (fun arg -> eval_term arg s i) args in
      RSet.mem arg_values (RMap.find p s.predicates)

## Checking whether $\mathcal{S}$ is a Group

In [ ]:
s

In [ ]:
Printf.printf "eval_formula g1 = %b\n" (eval_formula f1 s i);
Printf.printf "eval_formula g2 = %b\n" (eval_formula f2 s i);
Printf.printf "eval_formula g3 = %b\n" (eval_formula f3 s i);
Printf.printf "eval_formula g4 = %b\n" (eval_formula f4 s i)

## Another Example

Let's show that the formula $\forall X: \exists Y: p(X,Y) \rightarrow \exists Y: \forall X: p(X,Y)$ is not universally valid.

In [ ]:
let f_invalid = parse "∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)"

In [ ]:
let u2 : universe = RSet.of_list [0; 1]
let p_j : predicate_interpretation = RSet.of_list [[0; 1]; [1; 0]]
let s2 : structure = { universe   = u2;
                       functions  = RMap.empty;
                       predicates = RMap.of_list [("p", p_j)] }

In [ ]:
let i2 : variable_assignment = RMap.of_list [("X", 0); ("Y", 0)]

In [ ]:
Printf.printf "Result: %b\n" (eval_formula f_invalid s2 i2)